# Лабораторная работа 7. Анализ текста

Выполнил: Ашихмин Кирилл, группа P3124

Классификация твитов по тональности на корпусе RuTweetCorp.

Перед запуском данного борда необходимо сделать следующее:

0. Сделать копию борда через меню "Файл" / "File".
1. Выбрать пункт "Среда выполнения" или "Runtime" и выберите GPU в качестве аппаратного ускорителя в пункте меню "Сменить среду выполнения".


# Инструменты для работы с языком

... или зачем нужна предобработка.

Раньше мы смотрели на светлую сторону анализа данных - построение моделей. Теперь попробуем глубже посмотреть на часть про предобработку данных. Задача предобработки особенно актуальна, если мы имеем дело с текстами.

## Задача: классификация твитов по тональности

У нас есть выборка из твитов.
Нам известна эмоциональная окраска каждого твита из выборки: положительная или отрицательная. Задача состоит в построении модели, которая по тексту твита предсказывает его эмоциональную окраску.

Классификацию по тональности используют в рекомендательных системах, чтобы понять, понравилось ли людям кафе, кино, etc.

Скачиваем выборку ([источник](http://study.mokoron.com/)): [положительные](https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/positive.csv), [отрицательные]( https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/negative.csv).

In [1]:
# Скачиваем выборку. curl есть и в Google Colab, и в macOS/Linux.
!curl -sL -o positive.csv https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/positive.csv
!curl -sL -o negative.csv https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/negative.csv

In [2]:
import pandas as pd # библиотека для удобной работы с датафреймами
import numpy as np # библиотека для удобной работы со списками и матрицами

# библиотека, где реализованы основные алгоритмы машинного обучения
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [3]:
!head positive.csv





"408906761416867842";"1386325943";"JumpyAlex";"@irina_dyshkant Вот что значит страшилка :D







In [4]:
!tail negative.csv


"425137934443233281";"1390195756";"Sonya_Star_14";"У нас физ ра на улице
Пака линт:(









Откроем файлы и создадим массив из текстов и правильных меток для твитов.
Сначала идут положительные твиты, потом отрицательные.

In [5]:
# загружаем положительные твиты
# TODO #2
positive = pd.read_csv('positive.csv', sep=';', usecols=[3], names=['text'])
positive['label'] = ['positive'] * len(positive)
# загружаем отрицательные твиты
# TODO #3
negative = pd.read_csv('negative.csv', sep=';', usecols=[3], names=['text'])
negative['label'] = ['negative'] * len(negative)
# соединяем вместе
# TODO #4
df = pd.concat([positive, negative])


In [6]:
df

,text,label
0,"@first_timee хоть я и школота, но поверь, у на...",positive
1,"Да, все-таки он немного похож на него. Но мой ...",positive
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,positive
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",positive
4,@irina_dyshkant Вот что значит страшилка :D\nН...,positive
...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,negative
111919,скучаю так :-( только @taaannyaaa вправляет мо...,negative
111920,"Вот и в школу, в говно это идти уже надо(",negative
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",negative


Посмотрим на полученные данные:

In [7]:
df.sample(5, random_state=40)

,text,label
15931,RT @Blawar_1337: Теперь у нас с @Wake_UA появи...,positive
59532,с днём рождения зайка*))) ухх погуляем мы сего...,positive
47185,RT @Shumkova0406199: @ann_safina Вов вов вов А...,negative
42002,"Надо выдернуть звуковую дорожку из ""Доктора Ка...",positive
109035,@_hassliebe_ может все таки на этой неделе вер...,negative


Разбиваем данные на обучающую и тестовую выборки с помощью функции ```train_test_split()``` из **sklearn**:


In [8]:
# random_state фиксирует разбиение, чтобы при повторном запуске ноутбука
# получались те же самые числа в отчёте.
x_train, x_test, y_train, y_test = train_test_split(df.text, df.label, random_state=42)


print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(170125,) (56709,) (170125,) (56709,)


In [9]:
y_train[:10]

95791    negative
63858    positive
77643    positive
7876     negative
62141    positive
60237    positive
66071    negative
9607     positive
77938    negative
98365    positive
Name: label, dtype: str

In [10]:
y_train.value_counts()

label
positive    86310
negative    83815
Name: count, dtype: int64

## Baseline: классификация необработанных n-грамм

* Сейчас мы попробуем получить преобразование предложений в численный вектор, с которым может работать стандартный алгоритм машинного обучения, такой как логистическая регрессия.
* Для этого нам понадобится познакомиться с понятием n-gram - самых мелких элементов предложения, с которыми можно работать.
* Подсчитав количество этих n-грам в предложениях, мы получим искомые численные представления.

## Что такое n-граммы:

Самые мелкие структуры языка, с которыми мы работаем, называются **n-граммами**.
У n-граммы есть параметр n - количество слов, которые попадают в такое представление текста.
* Если n = 1 - то мы смотрим на то, сколько раз каждое слово встретилось в тексте. Получаем _униграммы_
* Если n = 2 - то мы смотрим на то, сколько раз каждая пара подряд идущих слов, встретилась в тексте. Получаем _биграммы_

**Функция** для работы с n-граммами реализована в библиотке **nltk** (Natural Language ToolKit), импортируем эту функцию:

In [11]:
from nltk import ngrams

Прежде чем получать n-граммы, нужно разделить предложение на отдельные слова.  Для этого используем метод ```split()```.

In [12]:
sentence = 'Если б мне платили каждый раз'.split()
sentence

['Если', 'б', 'мне', 'платили', 'каждый', 'раз']

Чтобы получить n-грамму для такой последовательности, используем функцию ```ngrams()```.

На вход передается два параметра:
* лист с разделенным на отдельные слова предложением (у нас он хранится в переменной ```sent```);
* параметр n, определяющий, какой тип n-грамм мы хотим получить.


Чтобы полученный объект отобразить, делаем из него ```list```.

In [13]:
list(ngrams(sentence, 1)) # униграммы

[('Если',), ('б',), ('мне',), ('платили',), ('каждый',), ('раз',)]

Аналогично мы можем получить биграммы - для этого заменяем параметр **n** в функции **ngrams** с 1 на 2.

In [14]:
list(ngrams(sentence, 2)) # биграммы

[('Если', 'б'),
 ('б', 'мне'),
 ('мне', 'платили'),
 ('платили', 'каждый'),
 ('каждый', 'раз')]

In [15]:
list(ngrams(sentence, 3)) # триграммы

[('Если', 'б', 'мне'),
 ('б', 'мне', 'платили'),
 ('мне', 'платили', 'каждый'),
 ('платили', 'каждый', 'раз')]

In [16]:
list(ngrams(sentence, 5)) # ... пентаграммы?

[('Если', 'б', 'мне', 'платили', 'каждый'),
 ('б', 'мне', 'платили', 'каждый', 'раз')]

### Векторизаторы

Векторизатор преобразует слово или набор слов в числовой вектор, понятный алгоритму машинного обучения, который привык работать с числовыми табличными данными.

Ниже - пример преобразования слов в двумерных вектор, каждому слову соответствует точка на плоскости.

<a href="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg
" target="_blank"><img src="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg"
alt="IMAGE ALT TEXT HERE" width="600" border="0" /></a>

На начальном этапе нам будет достаточно тех инструментов, которые уже есть в знакомой нам библиотеке **sklearn**.

In [17]:
from sklearn.linear_model import LogisticRegression # можно заменить на любимый классификатор
from sklearn.feature_extraction.text import CountVectorizer # модель "мешка слов", см. далее

Самый простой способ извлечь признаки из текстовых данных -- векторизаторы: `CountVectorizer` и `TfidfVectorizer`

Объект `CountVectorizer` делает следующую вещь:
* строит для каждого документа (каждой пришедшей ему строки) вектор размерности `n`, где `n` -- количество слов или n-грам во всём корпусе
* заполняет каждый i-тый элемент количеством вхождений слова в данный документ

<a href="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg
" target="_blank"><img src="https://drive.google.com/uc?id=1jHmkrGZTMawM46Yzxh243Ur1y5pYKzrl"
alt="IMAGE ALT TEXT HERE" width="600" border="0" /></a>

На рисунке пример векторизации для униграмм, но можно использовать любые n-граммы. Для этого у объекта ```CountVectorizer()``` есть параметр **ngram_range**, который отвечает за то, какие n-граммы мы используем в качестве признаов:<br/>
ngram_range=(1, 1) -- униграммы<br/>
ngram_range=(3, 3) -- триграммы<br/>
ngram_range=(1, 3) -- униграммы, биграммы и триграммы.

<a href="https://drive.google.com/uc?id=1ODNVK0fdLTX4nv6ob55ciUe37d1pio-D" target="_blank"><img src="https://drive.google.com/uc?id=1ODNVK0fdLTX4nv6ob55ciUe37d1pio-D"
alt="IMAGE ALT TEXT HERE" width="800" border="0" /></a>

Инициализируем ```CountVectorizer()```, указав в качестве признаков униграммы:

In [18]:
vectorizer = CountVectorizer(ngram_range=(1,1))

После инициализации _vectorizer_ можно обучить на наших данных.

Для обучения используем обучающую выборку ```x_train```, но в отличие от классификатора мы используем метод ```fit_transform()```: сначала обучаем наш векторизатор, а потом сразу применяем его к нашему набору данных. Это похоже на то, как мы работали с label encoderом и one-hot-encoderом.


In [19]:
# TODO #8
# fit_transform = обучить векторизатор на текстах (составить словарь всех слов)
# и сразу превратить эти тексты в числовую матрицу.
vectorized_x_train = vectorizer.fit_transform(x_train)

vectorized_x_train

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1849210 stored elements and shape (170125, 243585)>

In [20]:
vectorized_x_train = vectorizer.fit_transform(x_train)

Так как результат не зависит от порядка слов в текстах, то говорят, что такая модель представления текстов в виде векторов получается из *гипотезы представления текста как мешка слов*

В vectorizer.vocabulary_ лежит словарь, отображение слов в их индексы:

In [21]:
list(vectorizer.vocabulary_.items())[:10]

[('хватит', 232913),
 ('писать', 182291),
 ('такие', 219481),
 ('глупости', 119889),
 ('надо', 162235),
 ('правильно', 191630),
 ('речь', 203440),
 ('формулировать', 231251),
 ('отдохнуть', 175540),
 ('от', 175056)]

В нашей выборке 170125 текстов (твитов), в них встречается 243760 разных слов.

In [22]:
vectorized_x_train.shape

(170125, 243585)

Так как теперь у нас есть **численное представление** и набор входных признаков, то мы можем обучить модель логистической регрессии (или любую другую из тех, на которые мы смотрели раньше, например, случайный лес)

In [23]:
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(vectorized_x_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

С тестовыми данными нужно проделать то же самое, что и с данными для обучения: сделать из текстов вектора, которые можно передавать в классификатор для прогноза класса объекта.

У нас уже есть обученный векторизатор ```vectorizer```, поэтому используем метод ```transform()``` (просто применить его), а не ```fit_transform``` (обучить и применить).

In [24]:
vectorized_x_test = vectorizer.transform(x_test)

Как раньше, для получения прогноза у обученного классификатора используем метод ```predict()```.

С помощью функции ```classification_report()```, которая считает сразу несколько метрик качества классификации, посмотрим на то, насколько хорошо мы предсказываем положительную или отрицательную тональность твита .

In [25]:
pred = clf.predict(vectorized_x_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.76      0.77      0.77     28108
    positive       0.77      0.77      0.77     28601

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77     56709
weighted avg       0.77      0.77      0.77     56709



## Бонус*: триграммы

Попробуем сделать то же самое, используя в качестве признаков триграммы:

In [26]:
# TODO #12

# инициализируем векторайзер
vectorizer_3 = CountVectorizer(ngram_range=(3,3))
# обучаем его и сразу применяем к x_train
vectorized_x_train_3 = vectorizer_3.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(vectorized_x_train_3, y_train)
# применяем обученный векторизатор к тестовым данным
vectorized_x_test_3 = vectorizer_3.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred = clf.predict(vectorized_x_test_3)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.72      0.46      0.56     28108
    positive       0.61      0.82      0.70     28601

    accuracy                           0.64     56709
   macro avg       0.66      0.64      0.63     56709
weighted avg       0.66      0.64      0.63     56709



Как вы думаете, почему в результатах теперь такой разброс по сравнению с
униграммами?

Ответ. Твит — очень короткий текст, в среднем около 10 слов. Из него можно
составить всего 7-8 триграмм, и почти каждая такая триграмма встречается в
корпусе ровно один раз. Получается, что признаки, выученные на обучающей
выборке, почти не встречаются в тестовой: модели просто не на что опереться,
и она вынуждена угадывать. Это классическое проявление разреженности
(sparsity) при росте n.

Униграммы работают лучше именно потому, что отдельное слово («хорошо»,
«плохо») повторяется во многих твитах, и статистики по нему хватает.

## Бонус**: TF-IDF векторизация

`TfidfVectorizer` делает то же, что и `CountVectorizer`, но в качестве значений выдает **tf-idf** каждого слова.

Как считается tf-idf:

**TF (term frequency)** – относительная частотность слова в документе:
$$ TF(t,d) = \frac{n_{t}}{\sum_k n_{k}} $$

**IDF (inverse document frequency)** – обратная частота документов, в которых есть это слово:
$$ IDF(t, D) = \mbox{log} \frac{|D|}{|{d : t \in d}|} $$

Перемножаем их:
$$TFIDF(t, d, D) = TF(t,d) \times IDF(i, D)$$

Сакральный смысл: если слово часто встречается в одном документе, но в целом по корпусу встречается в небольшом
количестве документов, у него высокий TF-IDF.

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

Действуем аналогично, как с ```CountVectorizer()```:

In [28]:
# TODO #13

# инициализируем векторизатор, в качестве переменных используем униграммы
tfidfvectorizer = TfidfVectorizer(ngram_range=(1, 1))
# обучаем его и сразу применяем к x_train
tfidf_vectorized_x_train = tfidfvectorizer.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(tfidf_vectorized_x_train, y_train)
# применяем обученный векторизатор к тестовым данным
tfidf_vectorized_x_test = tfidfvectorizer.transform(x_test)

# получаем предсказания и выводим информацию о качестве
pred = clf.predict(tfidf_vectorized_x_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.77      0.73      0.75     28108
    positive       0.75      0.79      0.77     28601

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



In [29]:
# TODO #14

# инициализируем векторизатор, в качестве переменных используем пентаграммы
tfidfvectorizer_5 = TfidfVectorizer(ngram_range=(5, 5))

# обучаем его и сразу применяем к x_train
tfidf_vectorized_x_train_5 = tfidfvectorizer_5.fit_transform(x_train)

# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(tfidf_vectorized_x_train_5, y_train)

# применяем обученный векторизатор к тестовым данным
tfidf_vectorized_x_test_5 = tfidfvectorizer_5.transform(x_test)

# получаем предсказания и выводим информацию о качестве
pred = clf.predict(tfidf_vectorized_x_test_5)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.95      0.11      0.20     28108
    positive       0.53      0.99      0.69     28601

    accuracy                           0.56     56709
   macro avg       0.74      0.55      0.45     56709
weighted avg       0.74      0.56      0.45     56709



## Токенизация

Токенизировать - значит, поделить текст на части: слова, ключевые слова, фразы, символы и т.д., иными словами **токены**.

Самый наивный способ токенизировать текст - разделить с помощью функции `split()`. Но `split` упускает очень много всего, например, не отделяет пунктуацию от слов. Кроме этого, есть ещё много менее тривиальных проблем, поэтому лучше использовать готовые токенизаторы.

In [30]:
import nltk # уже знакомая нам библиотека nltk
from nltk.tokenize import word_tokenize # готовый токенизатор библиотеки nltk

Чтобы использовать токенизатор ```word_tokenize```, нужно сначала скачать данные для nltk о пунктуации и стоп-словах. Это просто требование nltk, поэтому, особо не задумываясь, запустите следующую ячейку:  

In [31]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/user/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Применим токенизацию:

In [32]:
example = 'Но не каждый хочет что-то исправлять:('
word_tokenize(example)

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять', ':', '(']

Если использовать просто ```split()```, то грустный смайлик :( не отделяется от слова "исправлять":

In [33]:
example.split()

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять:(']

В nltk вообще есть довольно много токенизаторов:

In [34]:
from nltk import tokenize
dir(tokenize)[:16]

['BlanklineTokenizer',
 'LegalitySyllableTokenizer',
 'LineTokenizer',
 'MWETokenizer',
 'NLTKWordTokenizer',
 'PunktSentenceTokenizer',
 'PunktTokenizer',
 'RegexpTokenizer',
 'ReppTokenizer',
 'SExprTokenizer',
 'SpaceTokenizer',
 'StanfordSegmenter',
 'SyllableTokenizer',
 'TabTokenizer',
 'TextTilingTokenizer',
 'ToktokTokenizer']

Они умеют выдавать индексы в строке для начала и конца каждого слова-токена:

In [35]:
wh_tok = tokenize.WhitespaceTokenizer()
list(wh_tok.span_tokenize(example))

[(0, 2), (3, 5), (6, 12), (13, 18), (19, 25), (26, 38)]

Некторые токенизаторы ведут себя специфично:

In [36]:
tokenize.TreebankWordTokenizer().tokenize("don't stop me")

['do', "n't", 'stop', 'me']

А некоторые -- вообще не для текста на естественном языке:

In [37]:
tokenize.SExprTokenizer().tokenize("(a (b c)) d e (f)")

['(a (b c))', 'd', 'e', '(f)']

**Правильный токенизатор подбирается исходя из требований задачи!**

## Стоп-слова и пунктуация

**Стоп-слова** - это слова, которые часто встречаются практически в любом тексте и ничего интересного не говорят о конретном документе. Для модели это просто шум. А шум нужно убирать. По аналогичной причине убирают и пунктуацию.

In [38]:
# импортируем стоп-слова из библиотеки nltk
from nltk.corpus import stopwords

# посмотрим на стоп-слова для русского языка
print(stopwords.words('russian'))

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

*Знаки* пунктуации лучше импортировать из модуля **String**. В нем хранятся различные наборы констант для работы со строками (пунктуация, алфавит и др.).

In [39]:
from string import punctuation
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

Объединим стоп-слова и знаки пунктуации вместе и запишем в переменную ```noise```:

In [40]:
noise = stopwords.words('russian') + list(punctuation)

Теперь нужно обучать нашу модель с учетом новых знаний про токенизацию и стоп-слова.

Для этого мы можем собрать новый векторизатор, передав ему на вход:
* какие n-граммы нам нужны, параметр **ngram_range**;
* какой токенизатор мы используем, параметр **tokenizer**;
* какие у нас стоп-слова, параметр **stop_words**.

*Напоминание:* мы используем готовый токенизатор ```word_tokenize```, а стоп-слова хранятся в переменной ```noise```

In [41]:
# инициализируем умный векторайзер
# TODO #16 с word_tokenize
smart_tokenizer = CountVectorizer(ngram_range=(1, 1), tokenizer=word_tokenize,
                                  stop_words=noise)

In [42]:
# TODO #17

# обучаем его и сразу применяем к x_train
smart_vectorized_x_train = smart_tokenizer.fit_transform(x_train)

# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(smart_vectorized_x_train, y_train)

# применяем обученный векторайзер к тестовым данным
smart_vectorized_x_test = smart_tokenizer.transform(x_test)

# получаем предсказания и выводим информацию о качестве
pred = clf.predict(smart_vectorized_x_test)
print(classification_report(y_test, pred))

/usr/local/lib/python3.13/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/usr/local/lib/python3.13/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['``'] not in stop_words.
  warnings.warn(


              precision    recall  f1-score   support

    negative       0.77      0.80      0.78     28108
    positive       0.80      0.76      0.78     28601

    accuracy                           0.78     56709
   macro avg       0.78      0.78      0.78     56709
weighted avg       0.78      0.78      0.78     56709



Получилось лучше: accuracy выше, а также заметно подрос recall у негативного класса.

Что ещё можно сделать?

## Бонус*: Лемматизация

**Лемматизация** – это сведение разных форм одного слова к начальной форме – **лемме**. Почему это хорошо?
* Во-первых, естественно рассматривать как отдельный признак каждое *слово*, а не каждую его отдельную форму.
* Во-вторых, некоторые стоп-слова стоят только в начальной форме, и без лематизации выкидываем мы только её.

Для русского есть хороших лемматизатор pymorphy.

### [Pymorphy](http://pymorphy2.readthedocs.io/en/latest/)
Это модуль на питоне, довольно быстрый и с кучей функций.

In [43]:
# устанавливаем pymorphy3
!pip install pymorphy3

zsh:1: command not found: pip


В pymorphy2 для морфологического анализа слов есть ```MorphAnalyzer()```:

In [44]:
from pymorphy3 import MorphAnalyzer
# создадим объект pymorphy3_analyzer импортированного класса

# TODO #18
pymorphy3_analyzer = MorphAnalyzer()
pymorphy3_analyzer

pymorphy3 работает с отдельными словами. Если дать ему на вход предложение - он его просто не лемматизирует, т.к. не понимает:

In [45]:
sent = ['Если', 'б', 'мне', 'платили', 'каждый', 'раз']
sent

['Если', 'б', 'мне', 'платили', 'каждый', 'раз']

Лемматизируем слово "платили" из предложения ```sent``` с помощью метода ```parse()```:

In [46]:
ana = pymorphy3_analyzer.parse(sent[3])
ana

[Parse(word='платили', tag=OpencorporaTag('VERB,impf,tran plur,past,indc'), normal_form='платить', score=1.0, methods_stack=((DictionaryAnalyzer(), 'платили', 2471, 10),))]

Выведем его нормальную форму:

In [47]:
# normal_form — начальная форма слова (лемма).
# parse() возвращает список разборов, отсортированных по вероятности,
# поэтому берём первый — самый вероятный.
ana[0].normal_form

'платить'

## О важности эксплоративного анализа

Но иногда пунктуация бывает и не шумом - главное отталкиваться от задачи. Что будет если вообще не убирать пунктуацию?

In [48]:
# TODO #19

# инициализируем умный векторайзер stop-words НЕ ИСПОЛЬЗУЕМ!
# Отличие от предыдущего варианта только одно: не передаём stop_words,
# поэтому знаки препинания остаются полноценными признаками.
tokenizer_with_punctuation = CountVectorizer(ngram_range=(1, 1), tokenizer=word_tokenize)

# обучаем его и сразу применяем к x_train
punctuation_x_train = tokenizer_with_punctuation.fit_transform(x_train)

# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(punctuation_x_train, y_train)

# применяем обученный векторайзер к тестовым данным
punctuation_x_test = tokenizer_with_punctuation.transform(x_test)

# получаем предсказания и выводим информацию о качестве
pred = clf.predict(punctuation_x_test)
print(classification_report(y_test, pred))

/usr/local/lib/python3.13/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


              precision    recall  f1-score   support

    negative       1.00      1.00      1.00     28108
    positive       1.00      1.00      1.00     28601

    accuracy                           1.00     56709
   macro avg       1.00      1.00      1.00     56709
weighted avg       1.00      1.00      1.00     56709



Шок! Стоило оставить пунктуацию — и все метрики равны 1. Как это получилось?
Среди неё были очень значимые токены (как вы думаете, какие?).

Ответ: скобки — это смайлики. Корпус RuTweetCorp размечался
автоматически, по наличию смайлика в тексте: твиты со скобкой `)`
записывались в положительные, со скобкой `(` — в отрицательные. Сам смайлик
при этом из текста не убрали.

Получается, что метка класса буквально записана в самом тексте. Модель не
научилась понимать тональность — она нашла «шпаргалку» и выучила одно простое
правило: «есть `)` — значит positive».

Это классический пример утечки целевой переменной (target leakage): в
признаках оказалась информация, напрямую выводящая ответ. На этих данных
метрика 1.0 не означает, что модель хорошая — она означает, что задача
испорчена. В реальном применении (например, на отзывах без смайликов) такая
модель работать не будет.

Посмотрим, как один из супер-значительных токенов справится с классификацией безо всякого машинного обучения:

In [49]:
# TODO #20

# Сначала посмотрим, какие токены оказались "супер-значительными".
# Посчитаем, в какой доле твитов каждого класса встречается скобка.
for symbol in [')', '(']:
    in_positive = df[df.label == 'positive'].text.str.contains(symbol, regex=False).mean()
    in_negative = df[df.label == 'negative'].text.str.contains(symbol, regex=False).mean()
    print(f"символ {symbol!r}: в положительных {in_positive:.3f}, "
          f"в отрицательных {in_negative:.3f}")

print()

# А теперь классификация одним правилом, без машинного обучения:
# есть ")" -> положительный твит, нет -> отрицательный.
rule_predictions = x_test.str.contains(')', regex=False).map(
    {True: 'positive', False: 'negative'})

print(classification_report(y_test, rule_predictions))

символ ')': в положительных 0.829, в отрицательных 0.000
символ '(': в положительных 0.000, в отрицательных 0.947



              precision    recall  f1-score   support

    negative       0.85      1.00      0.92     28108
    positive       1.00      0.83      0.91     28601

    accuracy                           0.91     56709
   macro avg       0.93      0.91      0.91     56709
weighted avg       0.93      0.91      0.91     56709



## Символьные n-граммы

Теперь в качестве фичей используем, например, униграммы символов. Для этого необходимо установить в ```CountVectorizer()``` параметр ```analyzer = 'char'```, то есть анализировать символы.

In [50]:
# TODO #21

# инициализируем векторайзер для символов
# analyzer='char' означает, что признаками будут не слова, а отдельные символы
char_vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 1))

# обучаем его и сразу применяем к x_train
char_x_train = char_vectorizer.fit_transform(x_train)

# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(char_x_train, y_train)

# применяем обученный векторайзер к тестовым данным
char_x_test = char_vectorizer.transform(x_test)

# получаем предсказания и выводим информацию о качестве
pred = clf.predict(char_x_test)
print(classification_report(y_test, pred))

print('Всего символов-признаков:', len(char_vectorizer.vocabulary_))

              precision    recall  f1-score   support

    negative       1.00      0.99      1.00     28108
    positive       0.99      1.00      1.00     28601

    accuracy                           1.00     56709
   macro avg       1.00      1.00      1.00     56709
weighted avg       1.00      1.00      1.00     56709

Всего символов-признаков: 329


Из предыдущего раздела уже понятно, почему на этих данных точность равна 1.

Символьные n-граммы используются, например, для задачи определения языка. Ещё одна замечательная особенность признаков-символов - для них не нужна токенизация и лемматизация, можно использовать такой подход для языков, у которых нет готовых анализаторов.

# Самостоятельная работа

1. Изучите материал, представленный в борде.
2. Выполните все ячейки и получите результаты.
3. Приведите результаты таблицы classification_report в под этим заданием для модели LogisticRegression
4. Примените 2 альтернативных использованному алгоритму для решения задачи классификации (для примера XGBClassifier и еще какой-то один) и получите результаты в таблице classification_report
5. Для XGBClassifier вам потребуется задать параметры
```learning_rate=0.1, n_estimators=1000, max_depth=5, min_child_weight=3, gamma=0.2, subsample=0.6, colsample_bytree=1.0, objective='binary:logistic', nthread=4, scale_pos_weight=1, seed=27```

6. В разделе TF-IDF векторизация по аналогии с униграммами и пентаграммами вычислите classification_report для биграмм, триграмм опубликуйте результаты в отчете и укажите изменилась ли точность f1-score при их использовании по сравнению с униграммами и пентаграммами.

## Пункты 3-5. Сравнение алгоритмов классификации

Сравнивать будем на базовых униграммах `CountVectorizer` — том самом
представлении, что и в разделе Baseline.

Почему именно на них: по умолчанию `CountVectorizer` использует шаблон
`\b\w\w+\b`, то есть берёт только слова из двух и более букв, а знаки
препинания отбрасывает. Значит, смайликов-подсказок в признаках нет, и модели
действительно решают задачу, а не пользуются утечкой.

Помимо логистической регрессии возьмём:

* XGBClassifier — градиентный бустинг над деревьями, с параметрами из задания;
* LinearSVC — линейный метод опорных векторов, классика для текстов;
* MultinomialNB — наивный байесовский классификатор, специально
  предназначенный для счётчиков слов.

In [51]:
import time

from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier

# XGBoost работает только с числовыми метками, поэтому переводим строки в 0/1.
y_train_binary = (y_train == 'positive').astype(int)
y_test_binary = (y_test == 'positive').astype(int)

print('Матрица признаков:', vectorized_x_train.shape)

Матрица признаков: (170125, 243585)


In [52]:
# --- Модель 1: логистическая регрессия (базовая, пункт 3) ---
start = time.time()
logistic_model = LogisticRegression(random_state=42, max_iter=1000)
logistic_model.fit(vectorized_x_train, y_train)
logistic_pred = logistic_model.predict(vectorized_x_test)

print(f'LogisticRegression, обучение заняло {time.time() - start:.1f} с')
print(classification_report(y_test, logistic_pred))

LogisticRegression, обучение заняло 3.2 с
              precision    recall  f1-score   support

    negative       0.76      0.77      0.77     28108
    positive       0.77      0.77      0.77     28601

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77     56709
weighted avg       0.77      0.77      0.77     56709



In [53]:
# --- Модель 2: XGBClassifier (параметры из задания, пункт 5) ---
start = time.time()
xgb_model = XGBClassifier(
    learning_rate=0.1, n_estimators=1000, max_depth=5, min_child_weight=3,
    gamma=0.2, subsample=0.6, colsample_bytree=1.0, objective='binary:logistic',
    nthread=4, scale_pos_weight=1, seed=27,
)
xgb_model.fit(vectorized_x_train, y_train_binary)
xgb_pred = xgb_model.predict(vectorized_x_test)

print(f'XGBClassifier, обучение заняло {time.time() - start:.1f} с')
print(classification_report(y_test_binary, xgb_pred,
                            target_names=['negative', 'positive']))

XGBClassifier, обучение заняло 31.8 с
              precision    recall  f1-score   support

    negative       0.75      0.68      0.71     28108
    positive       0.71      0.78      0.74     28601

    accuracy                           0.73     56709
   macro avg       0.73      0.73      0.73     56709
weighted avg       0.73      0.73      0.73     56709



In [54]:
# --- Модель 3: LinearSVC ---
start = time.time()
svc_model = LinearSVC(random_state=42)
svc_model.fit(vectorized_x_train, y_train)
svc_pred = svc_model.predict(vectorized_x_test)

print(f'LinearSVC, обучение заняло {time.time() - start:.1f} с')
print(classification_report(y_test, svc_pred))

LinearSVC, обучение заняло 16.0 с
              precision    recall  f1-score   support

    negative       0.73      0.77      0.75     28108
    positive       0.77      0.73      0.75     28601

    accuracy                           0.75     56709
   macro avg       0.75      0.75      0.75     56709
weighted avg       0.75      0.75      0.75     56709



In [55]:
# --- Модель 4: MultinomialNB ---
start = time.time()
nb_model = MultinomialNB()
nb_model.fit(vectorized_x_train, y_train)
nb_pred = nb_model.predict(vectorized_x_test)

print(f'MultinomialNB, обучение заняло {time.time() - start:.1f} с')
print(classification_report(y_test, nb_pred))

MultinomialNB, обучение заняло 0.2 с
              precision    recall  f1-score   support

    negative       0.74      0.79      0.76     28108
    positive       0.78      0.72      0.75     28601

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



### Отчёт по пунктам 3-5: сравнение алгоритмов

Все модели обучены на одних и тех же признаках — униграммы `CountVectorizer`
(243 585 признаков, 170 125 обучающих твитов).

| Модель | accuracy | f1 (macro) | Время обучения |
|---|---|---|---|
| LogisticRegression | 0.77 | 0.77 | 3.2 с |
| MultinomialNB | 0.76 | 0.76 | 0.2 с |
| LinearSVC | 0.75 | 0.75 | 16.0 с |
| XGBClassifier | 0.73 | 0.73 | 31.8 с |

Градиентный бустинг проиграл простой логистической
регрессии, и это не ошибка настройки, а закономерность.

Почему так. В лабораторных 4 и 5 на табличных данных бустинг уверенно
выигрывал. Здесь ситуация обратная, потому что у текстовых признаков другая
природа:

1. Признаков очень много, и они разреженные. 243 585 признаков при том,
   что в одном твите около 10 слов, то есть 99.996% значений в строке равны
   нулю. Дерево строит разбиения вида «слово N встретилось больше k раз»;
   на такой матрице каждое разбиение отсекает крошечную долю объектов, и
   дереву нужно очень много уровней, чтобы охватить словарь.

2. Линейная модель здесь естественна. Тональность текста хорошо
   описывается суммой вкладов отдельных слов: «отлично» добавляет к
   положительности, «ужасно» — к отрицательности. Логистическая регрессия
   именно это и делает — присваивает каждому слову вес. Ей не нужно
   «изобретать» взаимодействия признаков, которых в задаче почти нет.

3. Бустинг ограничен по числу используемых признаков. 1000 деревьев
   глубины 5 дают максимум около 31 000 разбиений — этого физически не хватает,
   чтобы задействовать весь словарь из 243 тысяч слов. Большая часть слов
   моделью просто не используется.

Наивный Байес (`MultinomialNB`) отстал от
логистической регрессии всего на 1 процентный пункт, но обучился за 0.2
секунды против 31.8 у XGBoost — в 160 раз быстрее. Он специально
спроектирован под счётчики слов и остаётся отличным быстрым базовым решением
для текстов.

Практический вывод: для текстовой классификации на мешке слов линейные
модели — не «упрощённый» вариант, а правильный выбор. Бустинг имеет смысл,
когда признаков немного и между ними есть сложные взаимодействия.

## Пункт 6. TF-IDF: биграммы и триграммы

Уже посчитаны униграммы и пентаграммы. Добавим биграммы `(2, 2)` и триграммы
`(3, 3)`, чтобы увидеть всю картину целиком.

In [56]:
ngram_results = {}

for ngram_range in [(1, 1), (2, 2), (3, 3), (5, 5)]:
    start = time.time()

    vectorizer_n = TfidfVectorizer(ngram_range=ngram_range)
    train_matrix = vectorizer_n.fit_transform(x_train)
    test_matrix = vectorizer_n.transform(x_test)

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(train_matrix, y_train)
    predictions = model.predict(test_matrix)

    ngram_results[ngram_range] = {
        'признаков': train_matrix.shape[1],
        'accuracy': round(accuracy_score(y_test, predictions), 4),
        'f1 (macro)': round(f1_score(y_test, predictions, average='macro'), 4),
        'время, с': round(time.time() - start, 1),
    }

    print(f'--- TF-IDF, ngram_range={ngram_range} ---')
    print(classification_report(y_test, predictions))

pd.DataFrame(ngram_results).T

--- TF-IDF, ngram_range=(1, 1) ---
              precision    recall  f1-score   support

    negative       0.77      0.73      0.75     28108
    positive       0.75      0.79      0.77     28601

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



--- TF-IDF, ngram_range=(2, 2) ---
              precision    recall  f1-score   support

    negative       0.72      0.66      0.69     28108
    positive       0.69      0.75      0.72     28601

    accuracy                           0.71     56709
   macro avg       0.71      0.71      0.71     56709
weighted avg       0.71      0.71      0.71     56709



--- TF-IDF, ngram_range=(3, 3) ---
              precision    recall  f1-score   support

    negative       0.73      0.45      0.56     28108
    positive       0.61      0.83      0.70     28601

    accuracy                           0.64     56709
   macro avg       0.67      0.64      0.63     56709
weighted avg       0.67      0.64      0.63     56709



--- TF-IDF, ngram_range=(5, 5) ---
              precision    recall  f1-score   support

    negative       0.95      0.11      0.20     28108
    positive       0.53      0.99      0.69     28601

    accuracy                           0.56     56709
   macro avg       0.74      0.55      0.45     56709
weighted avg       0.74      0.56      0.45     56709



,,признаков,accuracy,f1 (macro),"время, с"
1,1,243585.0,0.7604,0.7602,2.2
2,2,1003905.0,0.7076,0.7068,5.5
3,3,1329184.0,0.6437,0.6291,5.5
5,5,1135271.0,0.5563,0.4458,4.2


### Отчёт по пункту 6: TF-IDF с разными n-граммами

| n-граммы | Количество признаков | accuracy | f1 (macro) | Время |
|---|---|---|---|---|
| униграммы (1, 1) | 243 585 | 0.7604 | 0.7602 | 2.2 с |
| биграммы (2, 2) | 1 003 905 | 0.7076 | 0.7068 | 5.5 с |
| триграммы (3, 3) | 1 329 184 | 0.6437 | 0.6291 | 5.5 с |
| пентаграммы (5, 5) | 1 135 271 | 0.5563 | 0.4458 | 4.2 с |

Ответ на вопрос задания: да, f1-score изменился — он монотонно падает с
ростом n. От униграмм к пентаграммам f1 (macro) снижается с 0.76 до 0.45,
то есть почти до уровня случайного угадывания.

Почему качество падает. Средний твит — около 10 слов. Из него можно
составить:

* 10 униграмм,
* 9 биграмм,
* 8 триграмм,
* 6 пентаграмм.

Количество *возможных* сочетаний при этом растёт лавинообразно: признаков
становится в 4-5 раз больше (с 243 тысяч до 1.3 миллиона), а полезного сигнала
в каждом — меньше. Почти каждая триграмма встречается в корпусе один раз,
и в тестовой выборке она уже не появится. Модель обучается на признаках,
которых при проверке не существует. Это разреженность данных (data
sparsity) — основная проблема при росте n.

У пентаграмм recall отрицательного класса
всего 0.11 при precision 0.95. Это значит, что модель почти всё относит к
положительному классу, а отрицательным называет лишь те редкие случаи, где
уверена наверняка. Классический признак того, что признаков для решения не
хватает и модель «сдалась» в пользу одного класса.

Почему количество признаков у пентаграмм (1.13 млн) меньше, чем у триграмм
(1.33 млн)? Потому что из коротких твитов пентаграмму часто вообще нельзя
составить: если в твите 4 слова, он не даёт ни одной пентаграммы. Такие твиты
превращаются в пустые строки матрицы — ещё одна причина провала.

Когда n-граммы больше единицы всё-таки полезны: на длинных текстах и для
устойчивых словосочетаний, где важен порядок слов («не понравилось» против
«понравилось»). На коротких твитах выигрыша не будет. Компромисс на практике —
`ngram_range=(1, 2)`: униграммы дают основной сигнал, а биграммы добавляют
самые частые пары, не разрушая статистику.